# 🎭 Emotion-Tone + Sentiment Models (companion to the fake-news / bias notebook)
Trains two more DistilBERT classifiers and exports them as INT8 ONNX so the app no longer needs the pretrained
RoBERTa models:

| Model | Dataset | Labels |
|---|---|---|
| `emotion_tone` | GoEmotions (simplified), regrouped | calm / neutral / charged / inflammatory |
| `sentiment` | TweetEval sentiment | negative / neutral / positive |

Kaggle: **GPU on**, **Internet on**, then **Run → Restart session → Run all** (~10-15 min).
Output: `models_extra.zip` → extract into `backend/models/`.

> The tone labels are **derived** from GoEmotions' 27 emotions with the grouping in the next cell (the same intensity
> idea the app already used). It is a documented design choice, not a human-labelled "tone" dataset.

In [ ]:
!pip install -q "transformers>=4.46.0" "datasets>=3.0.0" evaluate accelerate scikit-learn onnx onnxruntime

In [ ]:
import os, json, shutil, inspect, warnings
warnings.filterwarnings("ignore", message=".*gather along dimension 0.*")
import torch, numpy as np, transformers
from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import evaluate

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

print("transformers", transformers.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")
TOK_KW = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"
MAX_LEN = 128
OUTPUT_DIR = "./trained_models_extra"; os.makedirs(OUTPUT_DIR, exist_ok=True)
BASE = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE)

acc, f1m = evaluate.load("accuracy"), evaluate.load("f1")
def compute_metrics(p):
    preds = np.argmax(p[0], axis=-1)
    return {"accuracy": acc.compute(predictions=preds, references=p[1])["accuracy"],
            "f1": f1m.compute(predictions=preds, references=p[1], average="weighted")["f1"]}

def train_model(name, ds, id2label, epochs=2, lr=3e-5):
    label2id = {v: k for k, v in id2label.items()}
    tok = ds.map(lambda ex: tokenizer(ex["text"], truncation=True, max_length=MAX_LEN), batched=True)
    tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE, num_labels=len(id2label), id2label=id2label, label2id=label2id)
    out = os.path.join(OUTPUT_DIR, name)
    args = TrainingArguments(output_dir=out, eval_strategy="epoch", save_strategy="epoch",
        learning_rate=lr, per_device_train_batch_size=64, per_device_eval_batch_size=128,
        num_train_epochs=epochs, weight_decay=0.01, fp16=torch.cuda.is_available(),
        save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="accuracy",
        logging_steps=100, report_to="none")
    tr = Trainer(model=model, args=args, train_dataset=tok["train"], eval_dataset=tok["validation"],
                 data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                 compute_metrics=compute_metrics, **{TOK_KW: tokenizer})
    tr.train()
    print(f"Held-out TEST metrics ({name}):", tr.evaluate(tok["test"], metric_key_prefix="test"))
    tr.save_model(out); tokenizer.save_pretrained(out)
    return out

---
## 🎭 Model 1: emotional tone (GoEmotions → 4 tone bands)

In [ ]:
TONE_LABELS = ["calm", "neutral", "charged", "inflammatory"]
INFLAMMATORY = {"anger", "disgust", "fear", "grief"}
CHARGED = {"annoyance", "disapproval", "disappointment", "nervousness", "surprise", "excitement",
           "desire", "sadness", "embarrassment", "remorse"}
NEUTRALISH = {"neutral", "realization", "confusion", "curiosity"}
# everything else (joy, love, gratitude, admiration, approval, optimism, pride, relief, amusement, caring) -> calm

raw = None
for repo, cfg in [("google-research-datasets/go_emotions", "simplified"), ("go_emotions", "simplified")]:
    try:
        raw = load_dataset(repo, cfg); print("GoEmotions from", repo); break
    except Exception as e:
        print("skipped", repo, repr(e)[:150])
if raw is None:
    raise RuntimeError("Could not load GoEmotions. Turn Internet ON.")
names = raw["train"].features["labels"].feature.names

def to_tone(ex):
    emos = {names[i] for i in ex["labels"]}
    if emos & INFLAMMATORY: t = 3
    elif emos & CHARGED: t = 2
    elif emos and not emos <= NEUTRALISH: t = 0
    else: t = 1
    return {"text": ex["text"], "labels": t}

emo_ds = DatasetDict({s: raw[s].map(to_tone, remove_columns=raw[s].column_names) for s in ["train", "validation", "test"]})
counts = np.bincount(emo_ds["train"]["labels"], minlength=4)
print("Tone label counts:", dict(zip(TONE_LABELS, counts.tolist())))
assert (counts > 200).all(), f"Tone grouping produced an empty/tiny class: {counts}"
emotion_dir = train_model("emotion_tone", emo_ds, dict(enumerate(TONE_LABELS)))

---
## 💬 Model 2: sentiment (TweetEval)

In [ ]:
SENT_LABELS = ["negative", "neutral", "positive"]
raw = None
for repo, cfg in [("cardiffnlp/tweet_eval", "sentiment"), ("tweet_eval", "sentiment")]:
    try:
        raw = load_dataset(repo, cfg); print("TweetEval from", repo); break
    except Exception as e:
        print("skipped", repo, repr(e)[:150])
if raw is None:
    raise RuntimeError("Could not load TweetEval sentiment. Turn Internet ON.")

sent_ds = DatasetDict({s: raw[s].rename_column("label", "labels") for s in ["train", "validation", "test"]})
counts = np.bincount(sent_ds["train"]["labels"], minlength=3)
print("Sentiment label counts:", dict(zip(SENT_LABELS, counts.tolist())))
assert (counts > 1000).all(), f"Unexpected label distribution: {counts}"
sentiment_dir = train_model("sentiment", sent_ds, dict(enumerate(SENT_LABELS)))

---
## ⚡ ONNX export + INT8 quantization

In [ ]:
import onnxruntime
from onnxruntime.quantization import quantize_dynamic, QuantType
EXPORT = "./onnx_extra"; os.makedirs(EXPORT, exist_ok=True)

def export_and_quantize(model_dir, name):
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, attn_implementation="eager").to("cpu").eval()
    raw_p, q_p = f"{EXPORT}/{name}_raw.onnx", f"{EXPORT}/{name}.onnx"
    d = tok("Sample text for export.", return_tensors="pt")
    kw = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"],
              dynamic_axes={"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}, "logits": {0: "b"}},
              opset_version=17)
    with torch.no_grad():
        try:
            torch.onnx.export(model, (d["input_ids"], d["attention_mask"]), raw_p, dynamo=False, **kw)
        except TypeError:
            torch.onnx.export(model, (d["input_ids"], d["attention_mask"]), raw_p, **kw)
    quantize_dynamic(raw_p, q_p, weight_type=QuantType.QInt8); os.remove(raw_p)
    print(f"{name}: {os.path.getsize(q_p)/1024**2:.1f} MB")
    tok.save_pretrained(f"{EXPORT}/{name}_tokenizer")
    json.dump({str(k): v for k, v in model.config.id2label.items()}, open(f"{EXPORT}/{name}_labels.json", "w"), indent=2)

export_and_quantize(emotion_dir, "emotion_tone")
export_and_quantize(sentiment_dir, "sentiment")

---
## 🧪 Check + bundle

In [ ]:
def sm(x):
    x = x - x.max(); e = np.exp(x); return e / e.sum()
tests = ["Critics called the plan a disaster and a betrayal, and protesters are furious.",
         "The committee will meet Tuesday to review the proposed budget.",
         "Volunteers were thrilled and grateful after the community event raised record funds."]
for name in ["emotion_tone", "sentiment"]:
    s = onnxruntime.InferenceSession(f"{EXPORT}/{name}.onnx", providers=["CPUExecutionProvider"])
    t = AutoTokenizer.from_pretrained(f"{EXPORT}/{name}_tokenizer")
    lab = {int(k): v for k, v in json.load(open(f"{EXPORT}/{name}_labels.json")).items()}
    for x in tests:
        i = t(x, truncation=True, max_length=MAX_LEN, return_tensors="np")
        p = sm(s.run(["logits"], {"input_ids": i["input_ids"].astype(np.int64), "attention_mask": i["attention_mask"].astype(np.int64)})[0][0])
        print(f"{name:13s}", {lab[j]: round(float(p[j]), 2) for j in range(len(p))}, "|", x[:45])

shutil.make_archive("models_extra", "zip", EXPORT)
print(f"Created models_extra.zip ({os.path.getsize('models_extra.zip')/1024**2:.1f} MB) - download it from the Output pane")